# Domain Shift Result Reproduction

This notebook reproduces the Benin-to-SA domain-shift experiments. It launches the relevant runs and reads AUROC/AUPRC for source performance, zero-shot SA transfer, finetuned SA performance, and Benin source retention. Update the path cells after launching fresh jobs; otherwise they point to the completed runs used for the summary.


In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import re
import shlex
import shutil
import subprocess

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ultrai").exists():
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")

SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
TRAIN_SCRIPT = REPO_ROOT / "train_job.sh"
FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
SIMCLR_SCRIPT = REPO_ROOT / "train_job_simclr.sh"
BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "train.yaml"
FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"

EXPECTED_SOURCE_AUROC = 0.8847
EXPECTED_SOURCE_AUPRC = 0.8724
REPRO_TOLERANCE = 0.02

# Slurm settings. Adjust here if you want to use a different partition or wall time.
GPUS_PER_JOB = 4
SLURM_PARTITION = "normal"
SLURM_TIME = "11:59:59"

print(f"Repo root: {REPO_ROOT}")
print(f"Scratch root: {SCRATCH_ROOT}")
print(f"Training script: {TRAIN_SCRIPT}")
print(f"Finetune script: {FINETUNE_SCRIPT}")
print(f"SimCLR script: {SIMCLR_SCRIPT}")
print(f"Base config: {BASE_CONFIG}")
print(f"Finetune base config: {FINETUNE_BASE_CONFIG}")

## 1. Launch Benin Source Training

Submit the five Benin source folds and the dependent aggregation job.


In [ ]:
if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"{timestamp}__train__benin__all-folds__hmv-mil-legacy-parity-repro"
RUN_ROOT = SCRATCH_ROOT / "runs" / "train" / "benin" / RUN_NAME

launch_cmd = [
    "bash",
    str(TRAIN_SCRIPT),
    "--dataset", "benin",
    "--fold", "all",
    "--config", str(BASE_CONFIG),
    "--run-name", RUN_NAME,
    "--run-root", str(RUN_ROOT),
    "--gpus-per-job", str(GPUS_PER_JOB),
    "--partition", SLURM_PARTITION,
    "--time", SLURM_TIME,
]

print("Launching:")
print(" ".join(shlex.quote(part) for part in launch_cmd))

completed = subprocess.run(
    launch_cmd,
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)

print("\nSTDOUT")
print(completed.stdout)
if completed.stderr:
    print("\nSTDERR")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"Training submission failed with exit code {completed.returncode}")

def parse_train_job_output(stdout: str) -> dict:
    info = {}
    for line in stdout.splitlines():
        match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
        if match:
            info[match.group(1)] = match.group(2).strip()
    return info

RUN_INFO = parse_train_job_output(completed.stdout)
RUN_INFO.setdefault("run_name", RUN_NAME)
RUN_INFO.setdefault("run_root", str(RUN_ROOT))
RUN_INFO["launch_command"] = " ".join(shlex.quote(part) for part in launch_cmd)

print("\nRun info")
print(json.dumps(RUN_INFO, indent=2))
print(f"\nSet SOURCE_BASELINE_RUN_ROOT to this path in the result cell if you restart the kernel:\n{RUN_INFO['run_root']}")

In [ ]:
SOURCE_BASELINE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro")

## 2. Read Source Baseline Results

Read the aggregate Benin source metrics from `SOURCE_BASELINE_RUN_ROOT`.


In [ ]:
summary_csv = SOURCE_BASELINE_RUN_ROOT / "final_results" / "tb_results" / "tb_test_metrics_summary.csv"
by_fold_csv = SOURCE_BASELINE_RUN_ROOT / "final_results" / "tb_results" / "tb_test_metrics_by_fold.csv"

def run_manual_aggregate_if_possible(run_root: Path) -> None:
    fold_summaries = [run_root / f"fold{fold}" / "final_results" / "tb_results" / "tb_metrics_summary.csv" for fold in range(5)]
    missing = [path for path in fold_summaries if not path.exists()]
    if missing:
        missing_preview = "\n".join(str(path) for path in missing[:5])
        raise RuntimeError(
            "Aggregate CSV is not ready yet, and at least one fold result is missing. "
            "The training or aggregation jobs are probably still running.\n"
            f"Missing examples:\n{missing_preview}"
        )

    aggregate_cmd = [
        "bash",
        str(TRAIN_SCRIPT),
        "--aggregate",
        "--dataset", "benin",
        "--run-root", str(run_root),
    ]
    print("Run-level aggregate CSV is missing, but all fold summaries exist. Running aggregation now:")
    print(" ".join(shlex.quote(part) for part in aggregate_cmd))
    subprocess.run(aggregate_cmd, cwd=REPO_ROOT, check=True)

if not summary_csv.exists():
    run_manual_aggregate_if_possible(SOURCE_BASELINE_RUN_ROOT)

if not summary_csv.exists():
    raise FileNotFoundError(f"Missing aggregate source test summary: {summary_csv}")
if not by_fold_csv.exists():
    raise FileNotFoundError(f"Missing aggregate fold source test metrics: {by_fold_csv}")

def read_csv_rows(path: Path) -> list[dict]:
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

summary_rows = read_csv_rows(summary_csv)
fold_rows = read_csv_rows(by_fold_csv)
if len(summary_rows) != 1:
    raise ValueError(f"Expected one summary row in {summary_csv}, found {len(summary_rows)}")

summary = summary_rows[0]
source_result = {
    "run_root": str(SOURCE_BASELINE_RUN_ROOT),
    "n_folds": int(float(summary["num_folds"])),
    "mean_auroc": float(summary["auc_mean"]),
    "std_auroc": float(summary["auc_std"]),
    "mean_auprc": float(summary["auprc_mean"]),
    "std_auprc": float(summary["auprc_std"]),
    "expected_auroc": EXPECTED_SOURCE_AUROC,
    "expected_auprc": EXPECTED_SOURCE_AUPRC,
}
source_result["delta_auroc"] = source_result["mean_auroc"] - EXPECTED_SOURCE_AUROC
source_result["delta_auprc"] = source_result["mean_auprc"] - EXPECTED_SOURCE_AUPRC

if pd is not None:
    display(pd.DataFrame(fold_rows))
    display(pd.DataFrame([source_result]))
else:
    print(json.dumps(fold_rows, indent=2))
    print(json.dumps(source_result, indent=2))

print(
    "Source baseline from fresh run: "
    f"AUROC {source_result['mean_auroc']:.4f} +/- {source_result['std_auroc']:.4f}, "
    f"AUPRC {source_result['mean_auprc']:.4f} +/- {source_result['std_auprc']:.4f}"
)
print(
    "Historical target: "
    f"AUROC {EXPECTED_SOURCE_AUROC:.4f}, AUPRC {EXPECTED_SOURCE_AUPRC:.4f}; "
    f"deltas: AUROC {source_result['delta_auroc']:+.4f}, AUPRC {source_result['delta_auprc']:+.4f}"
)

if abs(source_result["delta_auroc"]) > REPRO_TOLERANCE or abs(source_result["delta_auprc"]) > REPRO_TOLERANCE:
    raise AssertionError(
        "Fresh source baseline is outside the configured tolerance. "
        f"Tolerance={REPRO_TOLERANCE}; result={source_result}"
    )

In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_source_run_roc_pr

source_curve_svg = (
    SOURCE_BASELINE_RUN_ROOT
    / "final_results"
    / "tb_results"
    / "source_baseline_roc_pr_mean_ci.svg"
)
source_curve_summary = plot_source_run_roc_pr(
    SOURCE_BASELINE_RUN_ROOT,
    evaluation_label="Benin Source Baseline",
    title="Benin Source Baseline Test Set Performance",
    subtitle="Mean ROC and Precision-Recall Curves Across Five Folds",
    output_svg=source_curve_svg,
)

print(f"Saved source AUROC/AUPRC graph to {source_curve_svg}")
print(
    "Source baseline curves: "
    f"AUROC {source_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {source_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{source_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {source_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {source_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{source_curve_summary['auprc']['ci_upper']:.4f})"
)


## 3. Launch Plain Benin-to-SA Finetuning

Submit plain supervised SA finetuning from each Benin source fold.


In [ ]:
if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"


if "PASTE_SOURCE_BASELINE_RUN_ROOT" in str(SOURCE_BASELINE_RUN_ROOT):
    raise RuntimeError("Set SOURCE_BASELINE_RUN_ROOT to the completed Benin source run root.")

NORMAL_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
NORMAL_FINETUNE_TAG = "normal-balanceclip2pos4-lowlr-repro"
normal_ft_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def parse_key_value_output(stdout: str) -> dict:
    info = {}
    for line in stdout.splitlines():
        match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
        if match:
            info[match.group(1)] = match.group(2).strip()
    return info

NORMAL_FINETUNE_RUNS = []
for fold in range(5):
    source_checkpoint = SOURCE_BASELINE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = SOURCE_BASELINE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing source config for fold{fold}: {source_config}")

    run_name = f"{normal_ft_timestamp}__finetune__benin_to_sa__src-fold{fold}__{NORMAL_FINETUNE_TAG}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", NORMAL_FINETUNE_PRESET,
        "--domain-adaptation", "none",
        "--run-name", run_name,
        "--run-root", str(run_root),
    ]

    print(f"\nLaunching normal finetune fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Finetune submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = NORMAL_FINETUNE_PRESET
    NORMAL_FINETUNE_RUNS.append(run_info)

print("\nNormal finetune runs")
print(json.dumps(NORMAL_FINETUNE_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into NORMAL_FINETUNE_RUN_ROOTS_BY_FOLD in the readout cell.")

In [ ]:
NORMAL_FINETUNE_RUN_ROOTS_BY_FOLD = {
        0: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold0__normal-balanceclip2pos4-lowlr-repro"),
        1: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold1__normal-balanceclip2pos4-lowlr-repro"),
        2: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold2__normal-balanceclip2pos4-lowlr-repro"),
        3: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold3__normal-balanceclip2pos4-lowlr-repro"),
        4: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold4__normal-balanceclip2pos4-lowlr-repro"),
    }

## 4. Read Plain Finetuning Results

Read source, zero-shot SA, finetuned SA, and source-retention metrics for plain finetuning.


In [ ]:
from pathlib import Path
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc"))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

normal_ft_rows = []
for fold, run_root in sorted(NORMAL_FINETUNE_RUN_ROOTS_BY_FOLD.items()):
    zero_payload = load_json(run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json")
    target_payload = load_json(run_root / "results" / "finetune_full" / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    normal_ft_rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(normal_ft_rows, key)
    summary[f"{key}_mean"] = mean_value
    summary[f"{key}_std"] = std_value

try:
    summary["source_model_auroc"] = source_result["mean_auroc"]
    summary["source_model_auprc"] = source_result["mean_auprc"]
except NameError:
    summary["source_model_auroc"] = EXPECTED_SOURCE_AUROC
    summary["source_model_auprc"] = EXPECTED_SOURCE_AUPRC

summary["finetune_preset"] = globals().get("NORMAL_FINETUNE_PRESET", "balance_clip2_pos4_lowlr")
summary["n_folds"] = len(normal_ft_rows)

historical_normal = {
    "target_finetuned_auroc_mean": 0.7143,
    "target_finetuned_auprc_mean": 0.4174,
    "source_retention_auroc_mean": 0.5828,
    "source_retention_auprc_mean": 0.4410,
    "zero_shot_auroc_mean": 0.4912,
    "zero_shot_auprc_mean": 0.0724,
}
for key, expected in historical_normal.items():
    summary[f"{key}_historical"] = expected
    summary[f"{key}_delta"] = summary[key] - expected

if pd is not None:
    display(pd.DataFrame(normal_ft_rows))
    display(pd.DataFrame([summary]))
else:
    print(json.dumps(normal_ft_rows, indent=2))
    print(json.dumps(summary, indent=2))

print(
    "Normal finetune summary: "
    f"zero-shot SA {summary['zero_shot_auroc_mean']:.4f}/{summary['zero_shot_auprc_mean']:.4f}; "
    f"finetuned SA {summary['target_finetuned_auroc_mean']:.4f}/{summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {summary['source_retention_auroc_mean']:.4f}/{summary['source_retention_auprc_mean']:.4f}"
)

In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

normal_finetune_curve_summary = plot_run_roots_roc_pr(
    NORMAL_FINETUNE_RUN_ROOTS_BY_FOLD,
    'results',
    'finetune_full',
    'finetuned_full_patient_predictions.csv',
    evaluation_label='Plain SA Fine-tuned',
    title='Plain SA Fine-tuned Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='plain_finetune_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved Plain SA fine-tuned AUROC/AUPRC graph to {normal_finetune_curve_summary['output_svg']}")
print(
    'Plain SA fine-tuned' + " curves: "
    f"AUROC {normal_finetune_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {normal_finetune_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{normal_finetune_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {normal_finetune_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {normal_finetune_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{normal_finetune_curve_summary['auprc']['ci_upper']:.4f})"
)


## 5. Launch SimCLR Backbone Pretraining

Pretrain fold-specific SimCLR vision backbones on Benin and SA videos.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    SIMCLR_SCRIPT
except NameError:
    SIMCLR_SCRIPT = REPO_ROOT / "train_job_simclr.sh"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

SIMCLR_PRESET = "benin_sa_strong_patientaware"
SIMCLR_TAG = "strong-patientaware-depth15-repro"
SIMCLR_TIME_LIMIT = "12:00:00"
SIMCLR_SA_SPLIT_CSV = Path(
    "/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/"
    "20260426__finetune__benin_to_sa__src-fold0__sa-simclr-hmv-mil-balance-clip2-pos4-lowlr-freezeclip-v1/"
    "results/splits/split_full.csv"
)

if not SIMCLR_SCRIPT.exists():
    raise FileNotFoundError(f"Missing SimCLR launcher: {SIMCLR_SCRIPT}")
if not SIMCLR_SA_SPLIT_CSV.exists():
    raise FileNotFoundError(f"Missing SA split for combined SimCLR: {SIMCLR_SA_SPLIT_CSV}")

simclr_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SIMCLR_RUN_NAME = f"{simclr_timestamp}__simclr__benin_sa__all-folds__{SIMCLR_TAG}"
SIMCLR_RUN_ROOT = SCRATCH_ROOT / "runs" / "simclr" / "benin_sa" / SIMCLR_RUN_NAME

if SIMCLR_RUN_ROOT.exists():
    raise FileExistsError(f"Refusing to reuse an existing SimCLR run directory: {SIMCLR_RUN_ROOT}")

launch_cmd = [
    "bash",
    str(SIMCLR_SCRIPT),
    "--dataset", "benin_sa",
    "--preset", SIMCLR_PRESET,
    "--split-csv", str(SIMCLR_SA_SPLIT_CSV),
    "--time-limit", SIMCLR_TIME_LIMIT,
    "--run-name", SIMCLR_RUN_NAME,
    "--run-root", str(SIMCLR_RUN_ROOT),
]

print("Launching SimCLR backbone pretraining:")
print(" ".join(shlex.quote(part) for part in launch_cmd))
completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"SimCLR submission failed with exit code {completed.returncode}")

SIMCLR_RUN_INFO = parse_key_value_output(completed.stdout)
SIMCLR_RUN_INFO.setdefault("run_name", SIMCLR_RUN_NAME)
SIMCLR_RUN_INFO.setdefault("run_root", str(SIMCLR_RUN_ROOT))
SIMCLR_RUN_INFO["preset"] = SIMCLR_PRESET
SIMCLR_RUN_INFO["sa_split_csv"] = str(SIMCLR_SA_SPLIT_CSV)
SIMCLR_RUN_INFO["launch_command"] = " ".join(shlex.quote(part) for part in launch_cmd)

print("\nSimCLR run info")
print(json.dumps(SIMCLR_RUN_INFO, indent=2))
print("\nIf you restart the kernel, paste this path into SIMCLR_RUN_ROOT in the next cells:")
print(SIMCLR_RUN_ROOT)

In [ ]:
SIMCLR_RUN_ROOT = Path(
        "/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/simclr/benin_sa/20260507_000622__simclr__benin_sa__all-folds__strong-patientaware-depth15-repro"
    )

## 6. Read SimCLR Backbone Pretraining Results

Verify the SimCLR checkpoints and summarize the self-supervised training logs.


In [ ]:
from pathlib import Path
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected result file: {path}")
    with path.open() as handle:
        return json.load(handle)

simclr_rows = []
for fold in range(5):
    fold_root = SIMCLR_RUN_ROOT / f"fold{fold}"
    vision_checkpoint = fold_root / "vision_encoder_best.pt"
    metrics_path = fold_root / "metrics.json"
    if not vision_checkpoint.exists():
        raise FileNotFoundError(f"Missing SimCLR vision checkpoint for fold{fold}: {vision_checkpoint}")
    metrics = load_json(metrics_path)
    history = metrics.get("history") or []
    final_epoch = history[-1] if history else {}
    simclr_rows.append({
        "fold": fold,
        "run_root": str(fold_root),
        "vision_checkpoint": str(vision_checkpoint),
        "best_loss": metrics.get("best_loss"),
        "best_epoch": metrics.get("best_epoch"),
        "final_loss": final_epoch.get("loss"),
        "final_contrastive_top1": final_epoch.get("contrastive_top1"),
        "num_videos": metrics.get("num_videos"),
        "num_patients": metrics.get("num_patients"),
    })

def summarize_numeric(rows: list[dict], key: str) -> tuple[float | None, float | None]:
    values = [float(row[key]) for row in rows if row.get(key) is not None]
    if not values:
        return None, None
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

simclr_summary = {"run_root": str(SIMCLR_RUN_ROOT), "n_folds": len(simclr_rows)}
for key in ["best_loss", "best_epoch", "final_loss", "final_contrastive_top1", "num_videos", "num_patients"]:
    mean_value, std_value = summarize_numeric(simclr_rows, key)
    simclr_summary[f"{key}_mean"] = mean_value
    simclr_summary[f"{key}_std"] = std_value

SIMCLR_BACKBONE_RUN_ROOT = SIMCLR_RUN_ROOT

if pd is not None:
    display(pd.DataFrame(simclr_rows))
    display(pd.DataFrame([simclr_summary]))
else:
    print(json.dumps(simclr_rows, indent=2))
    print(json.dumps(simclr_summary, indent=2))

print(f"SimCLR backbone run root: {SIMCLR_BACKBONE_RUN_ROOT}")

## 7. Launch Benin Source Training From SimCLR Backbone

Train the supervised Benin source model from the fold-matched SimCLR backbones.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    TRAIN_SCRIPT
except NameError:
    TRAIN_SCRIPT = REPO_ROOT / "train_job.sh"
try:
    BASE_CONFIG
except NameError:
    BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "train.yaml"
try:
    GPUS_PER_JOB
except NameError:
    GPUS_PER_JOB = 4
try:
    SLURM_PARTITION
except NameError:
    SLURM_PARTITION = "normal"
try:
    SLURM_TIME
except NameError:
    SLURM_TIME = "11:59:59"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

try:
    SIMCLR_BACKBONE_RUN_ROOT = Path(SIMCLR_BACKBONE_RUN_ROOT)
except NameError:
    try:
        SIMCLR_BACKBONE_RUN_ROOT = Path(SIMCLR_RUN_ROOT)
    except NameError:
        SIMCLR_BACKBONE_RUN_ROOT = Path(
            "/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/simclr/benin_sa/"
            "20260501__simclr__benin_sa__all-folds__strong-patientaware-depth15-v1"
        )

missing_backbones = [
    SIMCLR_BACKBONE_RUN_ROOT / f"fold{fold}" / "vision_encoder_best.pt"
    for fold in range(5)
    if not (SIMCLR_BACKBONE_RUN_ROOT / f"fold{fold}" / "vision_encoder_best.pt").exists()
]
if missing_backbones:
    raise FileNotFoundError("Missing SimCLR backbone checkpoints:\n" + "\n".join(str(path) for path in missing_backbones))

SIMCLR_SOURCE_TAG = "benin-sa-strong-patientaware-simclr-repro"
simclr_source_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SIMCLR_SOURCE_RUN_NAME = f"{simclr_source_timestamp}__train__benin__all-folds__{SIMCLR_SOURCE_TAG}"
SIMCLR_SOURCE_RUN_ROOT = SCRATCH_ROOT / "runs" / "train" / "benin" / SIMCLR_SOURCE_RUN_NAME

if SIMCLR_SOURCE_RUN_ROOT.exists():
    raise FileExistsError(f"Refusing to reuse an existing source run directory: {SIMCLR_SOURCE_RUN_ROOT}")

launch_cmd = [
    "bash",
    str(TRAIN_SCRIPT),
    "--dataset", "benin",
    "--fold", "all",
    "--config", str(BASE_CONFIG),
    "--vision-weights", str(SIMCLR_BACKBONE_RUN_ROOT),
    "--run-name", SIMCLR_SOURCE_RUN_NAME,
    "--run-root", str(SIMCLR_SOURCE_RUN_ROOT),
    "--gpus-per-job", str(GPUS_PER_JOB),
    "--partition", SLURM_PARTITION,
    "--time", SLURM_TIME,
]

print("Launching Benin source training from SimCLR backbone:")
print(" ".join(shlex.quote(part) for part in launch_cmd))
completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"SimCLR source training submission failed with exit code {completed.returncode}")

SIMCLR_SOURCE_RUN_INFO = parse_key_value_output(completed.stdout)
SIMCLR_SOURCE_RUN_INFO.setdefault("run_name", SIMCLR_SOURCE_RUN_NAME)
SIMCLR_SOURCE_RUN_INFO.setdefault("run_root", str(SIMCLR_SOURCE_RUN_ROOT))
SIMCLR_SOURCE_RUN_INFO["vision_weights"] = str(SIMCLR_BACKBONE_RUN_ROOT)
SIMCLR_SOURCE_RUN_INFO["launch_command"] = " ".join(shlex.quote(part) for part in launch_cmd)

print("\nSimCLR-initialized source run info")
print(json.dumps(SIMCLR_SOURCE_RUN_INFO, indent=2))
print("\nIf you restart the kernel, paste this path into SIMCLR_SOURCE_RUN_ROOT in the next cells:")
print(SIMCLR_SOURCE_RUN_ROOT)

In [ ]:
SIMCLR_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260507_082951__train__benin__all-folds__benin-sa-strong-patientaware-simclr-repro")

## 8. Read SimCLR-Initialized Source Results

Read the Benin source metrics for the SimCLR-initialized source model.


In [ ]:
from pathlib import Path
import csv
import json
import shlex
import subprocess

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    TRAIN_SCRIPT
except NameError:
    TRAIN_SCRIPT = REPO_ROOT / "train_job.sh"

if "PASTE" in str(SIMCLR_SOURCE_RUN_ROOT):
    raise RuntimeError("Set SIMCLR_SOURCE_RUN_ROOT to the completed SimCLR-initialized source run root.")

def read_single_row_csv(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected CSV: {path}")
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 1:
        raise ValueError(f"Expected one row in {path}, found {len(rows)}")
    return rows[0]

summary_csv = SIMCLR_SOURCE_RUN_ROOT / "final_results" / "tb_results" / "tb_test_metrics_summary.csv"
by_fold_csv = SIMCLR_SOURCE_RUN_ROOT / "final_results" / "tb_results" / "tb_test_metrics_by_fold.csv"

if not summary_csv.exists():
    fold_summaries = [
        SIMCLR_SOURCE_RUN_ROOT / f"fold{fold}" / "final_results" / "tb_results" / "tb_metrics_summary.csv"
        for fold in range(5)
    ]
    if all(path.exists() for path in fold_summaries):
        aggregate_cmd = [
            "bash", str(TRAIN_SCRIPT),
            "--aggregate",
            "--dataset", "benin",
            "--run-root", str(SIMCLR_SOURCE_RUN_ROOT),
        ]
        print("Aggregate summary is missing, but fold results exist. Running:")
        print(" ".join(shlex.quote(part) for part in aggregate_cmd))
        completed = subprocess.run(aggregate_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
        print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        if completed.returncode != 0:
            raise RuntimeError(f"Aggregation failed with exit code {completed.returncode}")
    else:
        missing = [str(path) for path in fold_summaries if not path.exists()]
        raise FileNotFoundError("The aggregate summary is missing and not all fold summaries exist yet:\n" + "\n".join(missing))

source_summary = read_single_row_csv(summary_csv)
simclr_source_result = {
    "run_root": str(SIMCLR_SOURCE_RUN_ROOT),
    "mean_auroc": float(source_summary["auc_mean"]),
    "std_auroc": float(source_summary["auc_std"]),
    "mean_auprc": float(source_summary["auprc_mean"]),
    "std_auprc": float(source_summary["auprc_std"]),
    "num_folds": int(float(source_summary["num_folds"])),
    "summary_csv": str(summary_csv),
    "by_fold_csv": str(by_fold_csv),
}

SIMCLR_SOURCE_EXPECTED_AUROC = 0.8774
SIMCLR_SOURCE_EXPECTED_AUPRC = 0.8655
simclr_source_result["historical_auroc"] = SIMCLR_SOURCE_EXPECTED_AUROC
simclr_source_result["historical_auprc"] = SIMCLR_SOURCE_EXPECTED_AUPRC
simclr_source_result["auroc_delta"] = simclr_source_result["mean_auroc"] - SIMCLR_SOURCE_EXPECTED_AUROC
simclr_source_result["auprc_delta"] = simclr_source_result["mean_auprc"] - SIMCLR_SOURCE_EXPECTED_AUPRC

if pd is not None:
    display(pd.DataFrame([simclr_source_result]))
    if by_fold_csv.exists():
        display(pd.read_csv(by_fold_csv))
else:
    print(json.dumps(simclr_source_result, indent=2))

print(
    "SimCLR source summary: "
    f"AUROC {simclr_source_result['mean_auroc']:.4f} +/- {simclr_source_result['std_auroc']:.4f}; "
    f"AUPRC {simclr_source_result['mean_auprc']:.4f} +/- {simclr_source_result['std_auprc']:.4f}"
)

## 9. Launch SimCLR Benin-to-SA Finetuning

Submit plain SA finetuning from each SimCLR-initialized Benin source fold.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

try:
    SIMCLR_SOURCE_RUN_ROOT = Path(simclr_source_result["run_root"])
except NameError:
    try:
        SIMCLR_SOURCE_RUN_ROOT = Path(SIMCLR_SOURCE_RUN_ROOT)
    except NameError:
        SIMCLR_SOURCE_RUN_ROOT = Path(
            "/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/"
            "20260502__train__benin__all-folds__benin-sa-strong-patientaware-simclr-v1-hmv-mil-parity"
        )

SIMCLR_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
SIMCLR_FINETUNE_TAG = "simclr-strong-patientaware-balanceclip2pos4-freezeclip-repro"
SIMCLR_FINETUNE_RUNTIME = "local-venv"
SIMCLR_FINETUNE_LOCAL_VENV = Path("/users/lxflk/.venvs/ultatron")
SIMCLR_FINETUNE_FREEZE_BACKBONE = True
SIMCLR_FINETUNE_CLIP_UNFREEZE_LAST_N_LAYERS = 2

if SIMCLR_FINETUNE_RUNTIME == "local-venv" and not (SIMCLR_FINETUNE_LOCAL_VENV / "bin" / "python").exists():
    raise FileNotFoundError(f"Missing local finetune venv: {SIMCLR_FINETUNE_LOCAL_VENV}")

simclr_ft_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SIMCLR_FINETUNE_RUNS = []

for fold in range(5):
    source_checkpoint = SIMCLR_SOURCE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = SIMCLR_SOURCE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing SimCLR source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing SimCLR source config for fold{fold}: {source_config}")

    run_name = f"{simclr_ft_timestamp}__finetune__benin_to_sa__src-fold{fold}__{SIMCLR_FINETUNE_TAG}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", SIMCLR_FINETUNE_PRESET,
        "--clip-unfreeze-last-n-layers", str(SIMCLR_FINETUNE_CLIP_UNFREEZE_LAST_N_LAYERS),
        "--domain-adaptation", "none",
        "--runtime", SIMCLR_FINETUNE_RUNTIME,
        "--local-venv", str(SIMCLR_FINETUNE_LOCAL_VENV),
        "--run-name", run_name,
        "--run-root", str(run_root),
    ]
    if SIMCLR_FINETUNE_FREEZE_BACKBONE:
        launch_cmd.insert(launch_cmd.index("--domain-adaptation"), "--freeze-backbone")

    print(f"\nLaunching SimCLR finetune fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"SimCLR finetune submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = SIMCLR_FINETUNE_PRESET
    run_info["freeze_backbone"] = SIMCLR_FINETUNE_FREEZE_BACKBONE
    run_info["clip_unfreeze_last_n_layers"] = SIMCLR_FINETUNE_CLIP_UNFREEZE_LAST_N_LAYERS
    run_info["runtime"] = SIMCLR_FINETUNE_RUNTIME
    SIMCLR_FINETUNE_RUNS.append(run_info)

print("\nSimCLR finetune runs")
print(json.dumps(SIMCLR_FINETUNE_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD in the readout cell.")

In [ ]:
from pathlib import Path

# Update these paths after launching fresh SimCLR finetuning jobs.
SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD = {
    0: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_201449__finetune__benin_to_sa__src-fold0__simclr-strong-patientaware-balanceclip2pos4-freezeclip-repro"),
    1: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_201449__finetune__benin_to_sa__src-fold1__simclr-strong-patientaware-balanceclip2pos4-freezeclip-repro"),
    2: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_201449__finetune__benin_to_sa__src-fold2__simclr-strong-patientaware-balanceclip2pos4-freezeclip-repro"),
    3: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_201449__finetune__benin_to_sa__src-fold3__simclr-strong-patientaware-balanceclip2pos4-freezeclip-repro"),
    4: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_201449__finetune__benin_to_sa__src-fold4__simclr-strong-patientaware-balanceclip2pos4-freezeclip-repro"),
}


## 10. Read SimCLR Transfer Results

Read source, zero-shot SA, finetuned SA, and source-retention metrics for SimCLR transfer.


In [ ]:
from pathlib import Path
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

if "SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD" not in globals():
    if "SIMCLR_FINETUNE_RUNS" in globals():
        SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD = {
            int(run["fold"]): Path(run["run_root"])
            for run in SIMCLR_FINETUNE_RUNS
        }
    else:
        raise RuntimeError(
            "Define SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD in the path cell after the SimCLR finetune launch cell."
        )

if any("PASTE_FOLD" in str(path) for path in SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD.values()):
    raise RuntimeError("Set SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD to the five completed SimCLR finetune run roots.")

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc"))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

simclr_ft_rows = []
for fold, run_root in sorted(SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD.items()):
    zero_payload = load_json(run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json")
    target_payload = load_json(run_root / "results" / "finetune_full" / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    simclr_ft_rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

simclr_transfer_summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(simclr_ft_rows, key)
    simclr_transfer_summary[f"{key}_mean"] = mean_value
    simclr_transfer_summary[f"{key}_std"] = std_value

try:
    simclr_transfer_summary["source_model_auroc"] = simclr_source_result["mean_auroc"]
    simclr_transfer_summary["source_model_auprc"] = simclr_source_result["mean_auprc"]
except NameError:
    simclr_transfer_summary["source_model_auroc"] = 0.8774
    simclr_transfer_summary["source_model_auprc"] = 0.8655

simclr_transfer_summary["finetune_preset"] = globals().get("SIMCLR_FINETUNE_PRESET", "balance_clip2_pos4_lowlr")
simclr_transfer_summary["freeze_backbone"] = globals().get("SIMCLR_FINETUNE_FREEZE_BACKBONE", True)
simclr_transfer_summary["clip_unfreeze_last_n_layers"] = globals().get("SIMCLR_FINETUNE_CLIP_UNFREEZE_LAST_N_LAYERS", 2)
simclr_transfer_summary["n_folds"] = len(simclr_ft_rows)

historical_simclr = {
    "source_model_auroc": 0.8774,
    "source_model_auprc": 0.8655,
    "zero_shot_auroc_mean": 0.6197,
    "zero_shot_auprc_mean": 0.1129,
    "target_finetuned_auroc_mean": 0.8058,
    "target_finetuned_auprc_mean": 0.4004,
    "source_retention_auroc_mean": 0.8456,
    "source_retention_auprc_mean": 0.8039,
}
for key, expected in historical_simclr.items():
    simclr_transfer_summary[f"{key}_historical"] = expected
    simclr_transfer_summary[f"{key}_delta"] = simclr_transfer_summary[key] - expected

SIMCLR_TRANSFER_ROWS = simclr_ft_rows
SIMCLR_TRANSFER_SUMMARY = simclr_transfer_summary

if pd is not None:
    display(pd.DataFrame(simclr_ft_rows))
    display(pd.DataFrame([simclr_transfer_summary]))
else:
    print(json.dumps(simclr_ft_rows, indent=2))
    print(json.dumps(simclr_transfer_summary, indent=2))

print(
    "SimCLR transfer summary: "
    f"source {simclr_transfer_summary['source_model_auroc']:.4f}/{simclr_transfer_summary['source_model_auprc']:.4f}; "
    f"zero-shot SA {simclr_transfer_summary['zero_shot_auroc_mean']:.4f}/{simclr_transfer_summary['zero_shot_auprc_mean']:.4f}; "
    f"finetuned SA {simclr_transfer_summary['target_finetuned_auroc_mean']:.4f}/{simclr_transfer_summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {simclr_transfer_summary['source_retention_auroc_mean']:.4f}/{simclr_transfer_summary['source_retention_auprc_mean']:.4f}"
)


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

simclr_finetune_curve_summary = plot_run_roots_roc_pr(
    SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD,
    'results',
    'finetune_full',
    'finetuned_full_patient_predictions.csv',
    evaluation_label='SimCLR + SA Fine-tuned',
    title='SimCLR Transfer Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='simclr_finetune_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved SimCLR transfer AUROC/AUPRC graph to {simclr_finetune_curve_summary['output_svg']}")
print(
    'SimCLR transfer' + " curves: "
    f"AUROC {simclr_finetune_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {simclr_finetune_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{simclr_finetune_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {simclr_finetune_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {simclr_finetune_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{simclr_finetune_curve_summary['auprc']['ci_upper']:.4f})"
)


## 11. Launch DANN

Submit one raw DANN SA-finetuning job per Benin source fold.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

DANN_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro")
DANN_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
DANN_RUNTIME = "edf"
DANN_LOCAL_VENV = Path("/users/lxflk/.venvs/ultatron")
DANN_FREEZE_BACKBONE = True
DANN_CLIP_UNFREEZE_LAST_N_LAYERS = 2
DANN_CONFIG_NAME = "target_emphasis"
DANN_RUN_TAG = "dann-raw-target-emphasis-repro"
DANN_CONFIG_SETS = [
    'dann_lambda=0.75',
    'dann_domain_warmup_epochs=1',
    'dann_min_best_epoch=2',
    'dann_domain_loss_weight=0.5',
    'dann_source_task_weight=0.5',
    'dann_target_task_weight=2.0',
    'dann_domain_lr=0.00008',
]

if DANN_RUNTIME == "local-venv" and not (DANN_LOCAL_VENV / "bin" / "python").exists():
    raise FileNotFoundError(f"Missing local finetune venv: {DANN_LOCAL_VENV}")

dann_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DANN_RUNS = []

for fold in range(5):
    source_checkpoint = DANN_SOURCE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = DANN_SOURCE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing DANN source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing DANN source config for fold{fold}: {source_config}")

    run_name = f"{dann_timestamp}__finetune__benin_to_sa__src-fold{fold}__{DANN_RUN_TAG}__cfg-{DANN_CONFIG_NAME}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    if run_root.exists():
        raise FileExistsError(f"Refusing to reuse an existing DANN run directory: {run_root}")

    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", DANN_FINETUNE_PRESET,
        "--clip-unfreeze-last-n-layers", str(DANN_CLIP_UNFREEZE_LAST_N_LAYERS),
    ]
    if DANN_FREEZE_BACKBONE:
        launch_cmd.append("--freeze-backbone")
    launch_cmd.extend([
        "--domain-adaptation", "dann",
        "--runtime", DANN_RUNTIME,
        "--run-name", run_name,
        "--run-root", str(run_root),
    ])
    if DANN_RUNTIME == "local-venv":
        launch_cmd.extend(["--local-venv", str(DANN_LOCAL_VENV)])
    for config_set in DANN_CONFIG_SETS:
        launch_cmd.extend(["--set", config_set])

    print(f"\nLaunching DANN fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"DANN submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = DANN_FINETUNE_PRESET
    run_info["domain_adaptation"] = "dann"
    run_info["config_name"] = DANN_CONFIG_NAME
    run_info["config_sets"] = list(DANN_CONFIG_SETS)
    run_info["freeze_backbone"] = DANN_FREEZE_BACKBONE
    run_info["clip_unfreeze_last_n_layers"] = DANN_CLIP_UNFREEZE_LAST_N_LAYERS
    run_info["runtime"] = DANN_RUNTIME
    DANN_RUNS.append(run_info)

print("\nDANN runs")
print(json.dumps(DANN_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into DANN_RUN_ROOTS_BY_FOLD in the next path cell.")


## 12. DANN Result Paths

Define the DANN run roots used by the readout and summary cells.


In [ ]:
from pathlib import Path

DANN_SOURCE_RUN_ROOT = Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro')
DANN_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095718__finetune__benin_to_sa__src-fold0__dann-raw-target-emphasis-repro__cfg-target_emphasis'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095718__finetune__benin_to_sa__src-fold1__dann-raw-target-emphasis-repro__cfg-target_emphasis'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095718__finetune__benin_to_sa__src-fold2__dann-raw-target-emphasis-repro__cfg-target_emphasis'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095718__finetune__benin_to_sa__src-fold3__dann-raw-target-emphasis-repro__cfg-target_emphasis'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095718__finetune__benin_to_sa__src-fold4__dann-raw-target-emphasis-repro__cfg-target_emphasis'),
}
DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold0__normal-balanceclip2pos4-lowlr-repro'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold1__normal-balanceclip2pos4-lowlr-repro'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold2__normal-balanceclip2pos4-lowlr-repro'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold3__normal-balanceclip2pos4-lowlr-repro'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold4__normal-balanceclip2pos4-lowlr-repro'),
}


## 13. Read DANN Results

Read DANN target and source-retention metrics, using configured zero-shot references if needed.


In [ ]:
from pathlib import Path
import csv
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

for required_name in ["DANN_SOURCE_RUN_ROOT", "DANN_RUN_ROOTS_BY_FOLD", "DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD"]:
    if required_name not in globals():
        raise RuntimeError(f"Run the DANN Result Paths cell before this readout cell. Missing {required_name}.")

DANN_RESULT_SUBDIR = "dann_full"
DANN_CONFIG_NAME = "target_emphasis"
DANN_CONFIG_SETS = [
    'dann_lambda=0.75',
    'dann_domain_warmup_epochs=1',
    'dann_min_best_epoch=2',
    'dann_domain_loss_weight=0.5',
    'dann_source_task_weight=0.5',
    'dann_target_task_weight=2.0',
    'dann_domain_lr=0.00008',
]
DANN_HISTORICAL_REFERENCE = {'zero_shot_auroc_mean': 0.4925, 'zero_shot_auprc_mean': 0.0719, 'target_finetuned_auroc_mean': 0.698, 'target_finetuned_auprc_mean': 0.1748, 'source_retention_auroc_mean': 0.8154, 'source_retention_auprc_mean': 0.7202}

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc", metrics.get("auroc")))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

def read_source_model_summary(run_root: Path) -> tuple[float, float]:
    candidates = [
        run_root / "final_results" / "tb_results" / "tb_test_metrics_summary.csv",
        run_root / "final_results" / "tb_results" / "tb_metrics_summary.csv",
    ]
    summary_csv = next((path for path in candidates if path.exists()), None)
    if summary_csv is None:
        raise FileNotFoundError("Missing source-model aggregate summary CSV under " + str(run_root))
    with summary_csv.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 1:
        raise ValueError(f"Expected one source summary row in {summary_csv}, found {len(rows)}")
    row = rows[0]
    auroc = row.get("auc_mean", row.get("auroc_mean"))
    auprc = row.get("auprc_mean")
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find source AUROC/AUPRC columns in {summary_csv}: {sorted(row)}")
    return float(auroc), float(auprc)

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

rows = []
for fold, run_root in sorted(DANN_RUN_ROOTS_BY_FOLD.items()):
    target_payload = load_json(run_root / "results" / DANN_RESULT_SUBDIR / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_shot_path = run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
    zero_reference_used = False
    if not zero_shot_path.exists():
        reference_root = DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD.get(fold)
        if reference_root is None:
            raise FileNotFoundError(f"Missing zero-shot result and no reference root configured for fold{fold}: {zero_shot_path}")
        zero_shot_path = reference_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
        zero_reference_used = True
    zero_payload = load_json(zero_shot_path)

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "result_subdir": DANN_RESULT_SUBDIR,
        "zero_shot_reference_used": zero_reference_used,
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(rows, key)
    summary[f"{key}_mean"] = mean_value
    summary[f"{key}_std"] = std_value

source_auroc, source_auprc = read_source_model_summary(DANN_SOURCE_RUN_ROOT)
summary.update({
    "source_model_auroc": source_auroc,
    "source_model_auprc": source_auprc,
    "domain_adaptation": "dann",
    "config_name": DANN_CONFIG_NAME,
    "config_sets": "; ".join(DANN_CONFIG_SETS),
    "finetune_preset": "balance_clip2_pos4_lowlr",
    "freeze_backbone": True,
    "clip_unfreeze_last_n_layers": 2,
    "result_subdir": DANN_RESULT_SUBDIR,
    "n_folds": len(rows),
    "zero_shot_reference_folds": sum(1 for row in rows if row["zero_shot_reference_used"]),
})

for key, expected in DANN_HISTORICAL_REFERENCE.items():
    summary[f"{key}_historical"] = expected
    summary[f"{key}_delta"] = summary[key] - expected

DANN_TRANSFER_ROWS = rows
DANN_TRANSFER_SUMMARY = summary

if pd is not None:
    display(pd.DataFrame(rows))
    display(pd.DataFrame([summary]))
else:
    print(json.dumps(rows, indent=2))
    print(json.dumps(summary, indent=2))

print(
    "DANN summary: "
    f"source {summary['source_model_auroc']:.4f}/{summary['source_model_auprc']:.4f}; "
    f"zero-shot SA {summary['zero_shot_auroc_mean']:.4f}/{summary['zero_shot_auprc_mean']:.4f}; "
    f"finetuned SA {summary['target_finetuned_auroc_mean']:.4f}/{summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {summary['source_retention_auroc_mean']:.4f}/{summary['source_retention_auprc_mean']:.4f}"
)


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

dann_curve_summary = plot_run_roots_roc_pr(
    DANN_RUN_ROOTS_BY_FOLD,
    'results',
    DANN_RESULT_SUBDIR,
    f"{DANN_RESULT_SUBDIR}_patient_predictions.csv",
    evaluation_label='DANN',
    title='DANN Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='dann_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved DANN AUROC/AUPRC graph to {dann_curve_summary['output_svg']}")
print(
    'DANN' + " curves: "
    f"AUROC {dann_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {dann_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{dann_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {dann_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {dann_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{dann_curve_summary['auprc']['ci_upper']:.4f})"
)


## 14. Launch FixMatch

Submit one FixMatch SA-finetuning job per Benin source fold.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

FIXMATCH_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro")
FIXMATCH_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
FIXMATCH_RUNTIME = "edf"
FIXMATCH_LOCAL_VENV = Path("/users/lxflk/.venvs/ultatron")
FIXMATCH_FREEZE_BACKBONE = True
FIXMATCH_CLIP_UNFREEZE_LAST_N_LAYERS = 2
FIXMATCH_CONFIG_NAME = "conservative_highconf"
FIXMATCH_RUN_TAG = "fixmatch-conservative-highconf-repro"
FIXMATCH_CONFIG_SETS = [
    'fixmatch_confidence_threshold=0.97',
    'fixmatch_lambda_u=0.5',
    'fixmatch_unsup_warmup_epochs=3',
    'fixmatch_learning_rate=0.00003',
    'fixmatch_strong_intensity_jitter=0.1',
    'fixmatch_strong_noise_std=0.03',
    'fixmatch_strong_temporal_dropout_prob=0.1',
    'fixmatch_strong_cutout_prob=0.25',
    'fixmatch_strong_cutout_ratio=0.2',
]

if FIXMATCH_RUNTIME == "local-venv" and not (FIXMATCH_LOCAL_VENV / "bin" / "python").exists():
    raise FileNotFoundError(f"Missing local finetune venv: {FIXMATCH_LOCAL_VENV}")

fixmatch_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
FIXMATCH_RUNS = []

for fold in range(5):
    source_checkpoint = FIXMATCH_SOURCE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = FIXMATCH_SOURCE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing FixMatch source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing FixMatch source config for fold{fold}: {source_config}")

    run_name = f"{fixmatch_timestamp}__finetune__benin_to_sa__src-fold{fold}__{FIXMATCH_RUN_TAG}__cfg-{FIXMATCH_CONFIG_NAME}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    if run_root.exists():
        raise FileExistsError(f"Refusing to reuse an existing FixMatch run directory: {run_root}")

    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", FIXMATCH_FINETUNE_PRESET,
        "--clip-unfreeze-last-n-layers", str(FIXMATCH_CLIP_UNFREEZE_LAST_N_LAYERS),
    ]
    if FIXMATCH_FREEZE_BACKBONE:
        launch_cmd.append("--freeze-backbone")
    launch_cmd.extend([
        "--domain-adaptation", "fixmatch",
        "--runtime", FIXMATCH_RUNTIME,
        "--run-name", run_name,
        "--run-root", str(run_root),
    ])
    if FIXMATCH_RUNTIME == "local-venv":
        launch_cmd.extend(["--local-venv", str(FIXMATCH_LOCAL_VENV)])
    for config_set in FIXMATCH_CONFIG_SETS:
        launch_cmd.extend(["--set", config_set])

    print(f"\nLaunching FixMatch fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"FixMatch submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = FIXMATCH_FINETUNE_PRESET
    run_info["domain_adaptation"] = "fixmatch"
    run_info["config_name"] = FIXMATCH_CONFIG_NAME
    run_info["config_sets"] = list(FIXMATCH_CONFIG_SETS)
    run_info["freeze_backbone"] = FIXMATCH_FREEZE_BACKBONE
    run_info["clip_unfreeze_last_n_layers"] = FIXMATCH_CLIP_UNFREEZE_LAST_N_LAYERS
    run_info["runtime"] = FIXMATCH_RUNTIME
    FIXMATCH_RUNS.append(run_info)

print("\nFixMatch runs")
print(json.dumps(FIXMATCH_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into FIXMATCH_RUN_ROOTS_BY_FOLD in the next path cell.")


## 15. FixMatch Result Paths

Define the FixMatch run roots used by the readout and summary cells.


In [ ]:
from pathlib import Path

FIXMATCH_SOURCE_RUN_ROOT = Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro')
FIXMATCH_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095737__finetune__benin_to_sa__src-fold0__fixmatch-conservative-highconf-repro__cfg-conservative_highconf'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095737__finetune__benin_to_sa__src-fold1__fixmatch-conservative-highconf-repro__cfg-conservative_highconf'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095737__finetune__benin_to_sa__src-fold2__fixmatch-conservative-highconf-repro__cfg-conservative_highconf'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095737__finetune__benin_to_sa__src-fold3__fixmatch-conservative-highconf-repro__cfg-conservative_highconf'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095737__finetune__benin_to_sa__src-fold4__fixmatch-conservative-highconf-repro__cfg-conservative_highconf'),
}
FIXMATCH_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold0__normal-balanceclip2pos4-lowlr-repro'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold1__normal-balanceclip2pos4-lowlr-repro'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold2__normal-balanceclip2pos4-lowlr-repro'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold3__normal-balanceclip2pos4-lowlr-repro'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold4__normal-balanceclip2pos4-lowlr-repro'),
}


## 16. Read FixMatch Results

Read FixMatch target and source-retention metrics, using configured zero-shot references if needed.


In [ ]:
from pathlib import Path
import csv
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

for required_name in ["FIXMATCH_SOURCE_RUN_ROOT", "FIXMATCH_RUN_ROOTS_BY_FOLD", "FIXMATCH_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD"]:
    if required_name not in globals():
        raise RuntimeError(f"Run the FixMatch Result Paths cell before this readout cell. Missing {required_name}.")

FIXMATCH_RESULT_SUBDIR = "fixmatch_full"
FIXMATCH_CONFIG_NAME = "conservative_highconf"
FIXMATCH_CONFIG_SETS = [
    'fixmatch_confidence_threshold=0.97',
    'fixmatch_lambda_u=0.5',
    'fixmatch_unsup_warmup_epochs=3',
    'fixmatch_learning_rate=0.00003',
    'fixmatch_strong_intensity_jitter=0.1',
    'fixmatch_strong_noise_std=0.03',
    'fixmatch_strong_temporal_dropout_prob=0.1',
    'fixmatch_strong_cutout_prob=0.25',
    'fixmatch_strong_cutout_ratio=0.2',
]
FIXMATCH_HISTORICAL_REFERENCE = {'zero_shot_auroc_mean': 0.4925, 'zero_shot_auprc_mean': 0.0719, 'target_finetuned_auroc_mean': 0.7214, 'target_finetuned_auprc_mean': 0.1897, 'source_retention_auroc_mean': 0.4488, 'source_retention_auprc_mean': 0.3954}

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc", metrics.get("auroc")))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

def read_source_model_summary(run_root: Path) -> tuple[float, float]:
    candidates = [
        run_root / "final_results" / "tb_results" / "tb_test_metrics_summary.csv",
        run_root / "final_results" / "tb_results" / "tb_metrics_summary.csv",
    ]
    summary_csv = next((path for path in candidates if path.exists()), None)
    if summary_csv is None:
        raise FileNotFoundError("Missing source-model aggregate summary CSV under " + str(run_root))
    with summary_csv.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 1:
        raise ValueError(f"Expected one source summary row in {summary_csv}, found {len(rows)}")
    row = rows[0]
    auroc = row.get("auc_mean", row.get("auroc_mean"))
    auprc = row.get("auprc_mean")
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find source AUROC/AUPRC columns in {summary_csv}: {sorted(row)}")
    return float(auroc), float(auprc)

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

rows = []
for fold, run_root in sorted(FIXMATCH_RUN_ROOTS_BY_FOLD.items()):
    target_payload = load_json(run_root / "results" / FIXMATCH_RESULT_SUBDIR / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_shot_path = run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
    zero_reference_used = False
    if not zero_shot_path.exists():
        reference_root = FIXMATCH_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD.get(fold)
        if reference_root is None:
            raise FileNotFoundError(f"Missing zero-shot result and no reference root configured for fold{fold}: {zero_shot_path}")
        zero_shot_path = reference_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
        zero_reference_used = True
    zero_payload = load_json(zero_shot_path)

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "result_subdir": FIXMATCH_RESULT_SUBDIR,
        "zero_shot_reference_used": zero_reference_used,
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(rows, key)
    summary[f"{key}_mean"] = mean_value
    summary[f"{key}_std"] = std_value

source_auroc, source_auprc = read_source_model_summary(FIXMATCH_SOURCE_RUN_ROOT)
summary.update({
    "source_model_auroc": source_auroc,
    "source_model_auprc": source_auprc,
    "domain_adaptation": "fixmatch",
    "config_name": FIXMATCH_CONFIG_NAME,
    "config_sets": "; ".join(FIXMATCH_CONFIG_SETS),
    "finetune_preset": "balance_clip2_pos4_lowlr",
    "freeze_backbone": True,
    "clip_unfreeze_last_n_layers": 2,
    "result_subdir": FIXMATCH_RESULT_SUBDIR,
    "n_folds": len(rows),
    "zero_shot_reference_folds": sum(1 for row in rows if row["zero_shot_reference_used"]),
})

for key, expected in FIXMATCH_HISTORICAL_REFERENCE.items():
    summary[f"{key}_historical"] = expected
    summary[f"{key}_delta"] = summary[key] - expected

FIXMATCH_TRANSFER_ROWS = rows
FIXMATCH_TRANSFER_SUMMARY = summary

if pd is not None:
    display(pd.DataFrame(rows))
    display(pd.DataFrame([summary]))
else:
    print(json.dumps(rows, indent=2))
    print(json.dumps(summary, indent=2))

print(
    "FixMatch summary: "
    f"source {summary['source_model_auroc']:.4f}/{summary['source_model_auprc']:.4f}; "
    f"zero-shot SA {summary['zero_shot_auroc_mean']:.4f}/{summary['zero_shot_auprc_mean']:.4f}; "
    f"finetuned SA {summary['target_finetuned_auroc_mean']:.4f}/{summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {summary['source_retention_auroc_mean']:.4f}/{summary['source_retention_auprc_mean']:.4f}"
)


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

fixmatch_curve_summary = plot_run_roots_roc_pr(
    FIXMATCH_RUN_ROOTS_BY_FOLD,
    'results',
    FIXMATCH_RESULT_SUBDIR,
    f"{FIXMATCH_RESULT_SUBDIR}_patient_predictions.csv",
    evaluation_label='FixMatch',
    title='FixMatch Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='fixmatch_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved FixMatch AUROC/AUPRC graph to {fixmatch_curve_summary['output_svg']}")
print(
    'FixMatch' + " curves: "
    f"AUROC {fixmatch_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {fixmatch_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{fixmatch_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {fixmatch_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {fixmatch_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{fixmatch_curve_summary['auprc']['ci_upper']:.4f})"
)


## 17. Launch EWC

Submit one EWC SA-finetuning job per Benin source fold.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

EWC_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro")
EWC_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
EWC_RUNTIME = "edf"
EWC_LOCAL_VENV = Path("/users/lxflk/.venvs/ultatron")
EWC_FREEZE_BACKBONE = True
EWC_CLIP_UNFREEZE_LAST_N_LAYERS = 2
EWC_CONFIG_NAME = "lambda1000_unbalanced"
EWC_RUN_TAG = "ewc-lambda1000-unbalanced-repro"
EWC_CONFIG_SETS = [
    'ewc_lambda=1000',
    'ewc_oversample_target_positive_class=false',
]

if EWC_RUNTIME == "local-venv" and not (EWC_LOCAL_VENV / "bin" / "python").exists():
    raise FileNotFoundError(f"Missing local finetune venv: {EWC_LOCAL_VENV}")

ewc_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
EWC_RUNS = []

for fold in range(5):
    source_checkpoint = EWC_SOURCE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = EWC_SOURCE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing EWC source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing EWC source config for fold{fold}: {source_config}")

    run_name = f"{ewc_timestamp}__finetune__benin_to_sa__src-fold{fold}__{EWC_RUN_TAG}__cfg-{EWC_CONFIG_NAME}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    if run_root.exists():
        raise FileExistsError(f"Refusing to reuse an existing EWC run directory: {run_root}")

    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", EWC_FINETUNE_PRESET,
        "--clip-unfreeze-last-n-layers", str(EWC_CLIP_UNFREEZE_LAST_N_LAYERS),
    ]
    if EWC_FREEZE_BACKBONE:
        launch_cmd.append("--freeze-backbone")
    launch_cmd.extend([
        "--domain-adaptation", "ewc",
        "--runtime", EWC_RUNTIME,
        "--run-name", run_name,
        "--run-root", str(run_root),
    ])
    if EWC_RUNTIME == "local-venv":
        launch_cmd.extend(["--local-venv", str(EWC_LOCAL_VENV)])
    for config_set in EWC_CONFIG_SETS:
        launch_cmd.extend(["--set", config_set])

    print(f"\nLaunching EWC fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"EWC submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = EWC_FINETUNE_PRESET
    run_info["domain_adaptation"] = "ewc"
    run_info["config_name"] = EWC_CONFIG_NAME
    run_info["config_sets"] = list(EWC_CONFIG_SETS)
    run_info["freeze_backbone"] = EWC_FREEZE_BACKBONE
    run_info["clip_unfreeze_last_n_layers"] = EWC_CLIP_UNFREEZE_LAST_N_LAYERS
    run_info["runtime"] = EWC_RUNTIME
    EWC_RUNS.append(run_info)

print("\nEWC runs")
print(json.dumps(EWC_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into EWC_RUN_ROOTS_BY_FOLD in the next path cell.")


## 18. EWC Result Paths

Define the EWC run roots used by the readout and summary cells.


In [ ]:
from pathlib import Path

EWC_SOURCE_RUN_ROOT = Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro')
EWC_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095749__finetune__benin_to_sa__src-fold0__ewc-lambda1000-unbalanced-repro__cfg-lambda1000_unbalanced'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095749__finetune__benin_to_sa__src-fold1__ewc-lambda1000-unbalanced-repro__cfg-lambda1000_unbalanced'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095749__finetune__benin_to_sa__src-fold2__ewc-lambda1000-unbalanced-repro__cfg-lambda1000_unbalanced'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095749__finetune__benin_to_sa__src-fold3__ewc-lambda1000-unbalanced-repro__cfg-lambda1000_unbalanced'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095749__finetune__benin_to_sa__src-fold4__ewc-lambda1000-unbalanced-repro__cfg-lambda1000_unbalanced'),
}
EWC_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold0__normal-balanceclip2pos4-lowlr-repro'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold1__normal-balanceclip2pos4-lowlr-repro'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold2__normal-balanceclip2pos4-lowlr-repro'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold3__normal-balanceclip2pos4-lowlr-repro'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold4__normal-balanceclip2pos4-lowlr-repro'),
}


## 19. Read EWC Results

Read EWC target and source-retention metrics, using configured zero-shot references if needed.


In [ ]:
from pathlib import Path
import csv
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

for required_name in ["EWC_SOURCE_RUN_ROOT", "EWC_RUN_ROOTS_BY_FOLD", "EWC_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD"]:
    if required_name not in globals():
        raise RuntimeError(f"Run the EWC Result Paths cell before this readout cell. Missing {required_name}.")

EWC_RESULT_SUBDIR = "ewc_full"
EWC_CONFIG_NAME = "lambda1000_unbalanced"
EWC_CONFIG_SETS = [
    'ewc_lambda=1000',
    'ewc_oversample_target_positive_class=false',
]
EWC_HISTORICAL_REFERENCE = {'zero_shot_auroc_mean': 0.4925, 'zero_shot_auprc_mean': 0.0719, 'target_finetuned_auroc_mean': 0.816, 'target_finetuned_auprc_mean': 0.2263, 'source_retention_auroc_mean': 0.7694, 'source_retention_auprc_mean': 0.694}

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc", metrics.get("auroc")))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

def read_source_model_summary(run_root: Path) -> tuple[float, float]:
    candidates = [
        run_root / "final_results" / "tb_results" / "tb_test_metrics_summary.csv",
        run_root / "final_results" / "tb_results" / "tb_metrics_summary.csv",
    ]
    summary_csv = next((path for path in candidates if path.exists()), None)
    if summary_csv is None:
        raise FileNotFoundError("Missing source-model aggregate summary CSV under " + str(run_root))
    with summary_csv.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 1:
        raise ValueError(f"Expected one source summary row in {summary_csv}, found {len(rows)}")
    row = rows[0]
    auroc = row.get("auc_mean", row.get("auroc_mean"))
    auprc = row.get("auprc_mean")
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find source AUROC/AUPRC columns in {summary_csv}: {sorted(row)}")
    return float(auroc), float(auprc)

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

rows = []
for fold, run_root in sorted(EWC_RUN_ROOTS_BY_FOLD.items()):
    target_payload = load_json(run_root / "results" / EWC_RESULT_SUBDIR / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_shot_path = run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
    zero_reference_used = False
    if not zero_shot_path.exists():
        reference_root = EWC_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD.get(fold)
        if reference_root is None:
            raise FileNotFoundError(f"Missing zero-shot result and no reference root configured for fold{fold}: {zero_shot_path}")
        zero_shot_path = reference_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
        zero_reference_used = True
    zero_payload = load_json(zero_shot_path)

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "result_subdir": EWC_RESULT_SUBDIR,
        "zero_shot_reference_used": zero_reference_used,
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(rows, key)
    summary[f"{key}_mean"] = mean_value
    summary[f"{key}_std"] = std_value

source_auroc, source_auprc = read_source_model_summary(EWC_SOURCE_RUN_ROOT)
summary.update({
    "source_model_auroc": source_auroc,
    "source_model_auprc": source_auprc,
    "domain_adaptation": "ewc",
    "config_name": EWC_CONFIG_NAME,
    "config_sets": "; ".join(EWC_CONFIG_SETS),
    "finetune_preset": "balance_clip2_pos4_lowlr",
    "freeze_backbone": True,
    "clip_unfreeze_last_n_layers": 2,
    "result_subdir": EWC_RESULT_SUBDIR,
    "n_folds": len(rows),
    "zero_shot_reference_folds": sum(1 for row in rows if row["zero_shot_reference_used"]),
})

for key, expected in EWC_HISTORICAL_REFERENCE.items():
    summary[f"{key}_historical"] = expected
    summary[f"{key}_delta"] = summary[key] - expected

EWC_TRANSFER_ROWS = rows
EWC_TRANSFER_SUMMARY = summary

if pd is not None:
    display(pd.DataFrame(rows))
    display(pd.DataFrame([summary]))
else:
    print(json.dumps(rows, indent=2))
    print(json.dumps(summary, indent=2))

print(
    "EWC summary: "
    f"source {summary['source_model_auroc']:.4f}/{summary['source_model_auprc']:.4f}; "
    f"zero-shot SA {summary['zero_shot_auroc_mean']:.4f}/{summary['zero_shot_auprc_mean']:.4f}; "
    f"finetuned SA {summary['target_finetuned_auroc_mean']:.4f}/{summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {summary['source_retention_auroc_mean']:.4f}/{summary['source_retention_auprc_mean']:.4f}"
)


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

ewc_curve_summary = plot_run_roots_roc_pr(
    EWC_RUN_ROOTS_BY_FOLD,
    'results',
    EWC_RESULT_SUBDIR,
    f"{EWC_RESULT_SUBDIR}_patient_predictions.csv",
    evaluation_label='EWC',
    title='EWC Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='ewc_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved EWC AUROC/AUPRC graph to {ewc_curve_summary['output_svg']}")
print(
    'EWC' + " curves: "
    f"AUROC {ewc_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {ewc_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{ewc_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {ewc_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {ewc_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{ewc_curve_summary['auprc']['ci_upper']:.4f})"
)


## 20. Launch LwF

Submit one LwF SA-finetuning job per Benin source fold.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

LWF_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro")
LWF_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
LWF_RUNTIME = "edf"
LWF_LOCAL_VENV = Path("/users/lxflk/.venvs/ultatron")
LWF_FREEZE_BACKBONE = True
LWF_CLIP_UNFREEZE_LAST_N_LAYERS = 2
LWF_CONFIG_NAME = "task_kd_0p03_t2"
LWF_RUN_TAG = "lwf-task-kd-0p03-t2-repro"
LWF_CONFIG_SETS = [
    'lwf_lambda=0.03',
    'lwf_temperature=2.0',
    'lwf_pathology_lambda=0.0',
    'lwf_distill_tasks=[]',
]

if LWF_RUNTIME == "local-venv" and not (LWF_LOCAL_VENV / "bin" / "python").exists():
    raise FileNotFoundError(f"Missing local finetune venv: {LWF_LOCAL_VENV}")

lwf_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LWF_RUNS = []

for fold in range(5):
    source_checkpoint = LWF_SOURCE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = LWF_SOURCE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing LwF source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing LwF source config for fold{fold}: {source_config}")

    run_name = f"{lwf_timestamp}__finetune__benin_to_sa__src-fold{fold}__{LWF_RUN_TAG}__cfg-{LWF_CONFIG_NAME}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    if run_root.exists():
        raise FileExistsError(f"Refusing to reuse an existing LwF run directory: {run_root}")

    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", LWF_FINETUNE_PRESET,
        "--clip-unfreeze-last-n-layers", str(LWF_CLIP_UNFREEZE_LAST_N_LAYERS),
    ]
    if LWF_FREEZE_BACKBONE:
        launch_cmd.append("--freeze-backbone")
    launch_cmd.extend([
        "--domain-adaptation", "lwf",
        "--runtime", LWF_RUNTIME,
        "--run-name", run_name,
        "--run-root", str(run_root),
    ])
    if LWF_RUNTIME == "local-venv":
        launch_cmd.extend(["--local-venv", str(LWF_LOCAL_VENV)])
    for config_set in LWF_CONFIG_SETS:
        launch_cmd.extend(["--set", config_set])

    print(f"\nLaunching LwF fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"LwF submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = LWF_FINETUNE_PRESET
    run_info["domain_adaptation"] = "lwf"
    run_info["config_name"] = LWF_CONFIG_NAME
    run_info["config_sets"] = list(LWF_CONFIG_SETS)
    run_info["freeze_backbone"] = LWF_FREEZE_BACKBONE
    run_info["clip_unfreeze_last_n_layers"] = LWF_CLIP_UNFREEZE_LAST_N_LAYERS
    run_info["runtime"] = LWF_RUNTIME
    LWF_RUNS.append(run_info)

print("\nLwF runs")
print(json.dumps(LWF_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into LWF_RUN_ROOTS_BY_FOLD in the next path cell.")


## 21. LwF Result Paths

Define the LwF run roots used by the readout and summary cells.


In [ ]:
from pathlib import Path

LWF_SOURCE_RUN_ROOT = Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260506_090813__train__benin__all-folds__hmv-mil-legacy-parity-repro')
LWF_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095809__finetune__benin_to_sa__src-fold0__lwf-task-kd-0p03-t2-repro__cfg-task_kd_0p03_t2'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095809__finetune__benin_to_sa__src-fold1__lwf-task-kd-0p03-t2-repro__cfg-task_kd_0p03_t2'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095809__finetune__benin_to_sa__src-fold2__lwf-task-kd-0p03-t2-repro__cfg-task_kd_0p03_t2'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095809__finetune__benin_to_sa__src-fold3__lwf-task-kd-0p03-t2-repro__cfg-task_kd_0p03_t2'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260507_095809__finetune__benin_to_sa__src-fold4__lwf-task-kd-0p03-t2-repro__cfg-task_kd_0p03_t2'),
}
LWF_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD = {
    0: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold0__normal-balanceclip2pos4-lowlr-repro'),
    1: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold1__normal-balanceclip2pos4-lowlr-repro'),
    2: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold2__normal-balanceclip2pos4-lowlr-repro'),
    3: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold3__normal-balanceclip2pos4-lowlr-repro'),
    4: Path('/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260506_171838__finetune__benin_to_sa__src-fold4__normal-balanceclip2pos4-lowlr-repro'),
}


## 22. Read LwF Results

Read LwF target and source-retention metrics, using configured zero-shot references if needed.


In [ ]:
from pathlib import Path
import csv
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

for required_name in ["LWF_SOURCE_RUN_ROOT", "LWF_RUN_ROOTS_BY_FOLD", "LWF_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD"]:
    if required_name not in globals():
        raise RuntimeError(f"Run the LwF Result Paths cell before this readout cell. Missing {required_name}.")

LWF_RESULT_SUBDIR = "lwf_full"
LWF_CONFIG_NAME = "task_kd_0p03_t2"
LWF_CONFIG_SETS = [
    'lwf_lambda=0.03',
    'lwf_temperature=2.0',
    'lwf_pathology_lambda=0.0',
    'lwf_distill_tasks=[]',
]
LWF_HISTORICAL_REFERENCE = {'zero_shot_auroc_mean': 0.4925, 'zero_shot_auprc_mean': 0.0719, 'target_finetuned_auroc_mean': 0.7286, 'target_finetuned_auprc_mean': 0.2067, 'source_retention_auroc_mean': 0.5871, 'source_retention_auprc_mean': 0.4811}

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc", metrics.get("auroc")))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

def read_source_model_summary(run_root: Path) -> tuple[float, float]:
    candidates = [
        run_root / "final_results" / "tb_results" / "tb_test_metrics_summary.csv",
        run_root / "final_results" / "tb_results" / "tb_metrics_summary.csv",
    ]
    summary_csv = next((path for path in candidates if path.exists()), None)
    if summary_csv is None:
        raise FileNotFoundError("Missing source-model aggregate summary CSV under " + str(run_root))
    with summary_csv.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 1:
        raise ValueError(f"Expected one source summary row in {summary_csv}, found {len(rows)}")
    row = rows[0]
    auroc = row.get("auc_mean", row.get("auroc_mean"))
    auprc = row.get("auprc_mean")
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find source AUROC/AUPRC columns in {summary_csv}: {sorted(row)}")
    return float(auroc), float(auprc)

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

rows = []
for fold, run_root in sorted(LWF_RUN_ROOTS_BY_FOLD.items()):
    target_payload = load_json(run_root / "results" / LWF_RESULT_SUBDIR / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_shot_path = run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
    zero_reference_used = False
    if not zero_shot_path.exists():
        reference_root = LWF_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD.get(fold)
        if reference_root is None:
            raise FileNotFoundError(f"Missing zero-shot result and no reference root configured for fold{fold}: {zero_shot_path}")
        zero_shot_path = reference_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
        zero_reference_used = True
    zero_payload = load_json(zero_shot_path)

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "result_subdir": LWF_RESULT_SUBDIR,
        "zero_shot_reference_used": zero_reference_used,
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(rows, key)
    summary[f"{key}_mean"] = mean_value
    summary[f"{key}_std"] = std_value

source_auroc, source_auprc = read_source_model_summary(LWF_SOURCE_RUN_ROOT)
summary.update({
    "source_model_auroc": source_auroc,
    "source_model_auprc": source_auprc,
    "domain_adaptation": "lwf",
    "config_name": LWF_CONFIG_NAME,
    "config_sets": "; ".join(LWF_CONFIG_SETS),
    "finetune_preset": "balance_clip2_pos4_lowlr",
    "freeze_backbone": True,
    "clip_unfreeze_last_n_layers": 2,
    "result_subdir": LWF_RESULT_SUBDIR,
    "n_folds": len(rows),
    "zero_shot_reference_folds": sum(1 for row in rows if row["zero_shot_reference_used"]),
})

for key, expected in LWF_HISTORICAL_REFERENCE.items():
    summary[f"{key}_historical"] = expected
    summary[f"{key}_delta"] = summary[key] - expected

LWF_TRANSFER_ROWS = rows
LWF_TRANSFER_SUMMARY = summary

if pd is not None:
    display(pd.DataFrame(rows))
    display(pd.DataFrame([summary]))
else:
    print(json.dumps(rows, indent=2))
    print(json.dumps(summary, indent=2))

print(
    "LwF summary: "
    f"source {summary['source_model_auroc']:.4f}/{summary['source_model_auprc']:.4f}; "
    f"zero-shot SA {summary['zero_shot_auroc_mean']:.4f}/{summary['zero_shot_auprc_mean']:.4f}; "
    f"finetuned SA {summary['target_finetuned_auroc_mean']:.4f}/{summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {summary['source_retention_auroc_mean']:.4f}/{summary['source_retention_auprc_mean']:.4f}"
)


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

lwf_curve_summary = plot_run_roots_roc_pr(
    LWF_RUN_ROOTS_BY_FOLD,
    'results',
    LWF_RESULT_SUBDIR,
    f"{LWF_RESULT_SUBDIR}_patient_predictions.csv",
    evaluation_label='LwF',
    title='LwF Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='lwf_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved LwF AUROC/AUPRC graph to {lwf_curve_summary['output_svg']}")
print(
    'LwF' + " curves: "
    f"AUROC {lwf_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {lwf_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{lwf_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {lwf_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {lwf_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{lwf_curve_summary['auprc']['ci_upper']:.4f})"
)


## 23. Launch SimCLR+DANN Finetuning

Submit DANN SA finetuning from each SimCLR-initialized Benin source fold.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shlex
import shutil
import subprocess

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available in this environment. Run this cell on the CSCS login environment.")

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path("/users/lxflk/ULTR-AI-Vid")
try:
    SCRATCH_ROOT
except NameError:
    SCRATCH_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid")
try:
    FINETUNE_SCRIPT
except NameError:
    FINETUNE_SCRIPT = REPO_ROOT / "train_job_finetune.sh"
try:
    FINETUNE_BASE_CONFIG
except NameError:
    FINETUNE_BASE_CONFIG = REPO_ROOT / "configs" / "cscs" / "finetune.yaml"
try:
    SIMCLR_SOURCE_RUN_ROOT
except NameError:
    SIMCLR_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260507_082951__train__benin__all-folds__benin-sa-strong-patientaware-simclr-repro")

try:
    parse_key_value_output
except NameError:
    def parse_key_value_output(stdout: str) -> dict:
        info = {}
        for line in stdout.splitlines():
            match = re.match(r"\s*([A-Za-z_]+):\s*(.*)$", line)
            if match:
                info[match.group(1)] = match.group(2).strip()
        return info

SIMCLR_DANN_SOURCE_RUN_ROOT = Path(SIMCLR_SOURCE_RUN_ROOT)
SIMCLR_DANN_FINETUNE_PRESET = "balance_clip2_pos4_lowlr"
SIMCLR_DANN_RUNTIME = "edf"
SIMCLR_DANN_LOCAL_VENV = Path("/users/lxflk/.venvs/ultatron")
SIMCLR_DANN_FREEZE_BACKBONE = True
SIMCLR_DANN_CLIP_UNFREEZE_LAST_N_LAYERS = 2
SIMCLR_DANN_CONFIG_NAME = "target_emphasis"
SIMCLR_DANN_RUN_TAG = "simclr-dann-target-emphasis-repro"
SIMCLR_DANN_CONFIG_SETS = [
    'dann_lambda=0.75',
    'dann_domain_warmup_epochs=1',
    'dann_min_best_epoch=2',
    'dann_domain_loss_weight=0.5',
    'dann_source_task_weight=0.5',
    'dann_target_task_weight=2.0',
    'dann_domain_lr=0.00008',
]

if SIMCLR_DANN_RUNTIME == "local-venv" and not (SIMCLR_DANN_LOCAL_VENV / "bin" / "python").exists():
    raise FileNotFoundError(f"Missing local finetune venv: {SIMCLR_DANN_LOCAL_VENV}")

simclr_dann_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
SIMCLR_DANN_RUNS = []

for fold in range(5):
    source_checkpoint = SIMCLR_DANN_SOURCE_RUN_ROOT / f"fold{fold}" / "checkpoint_best.pth"
    source_config = SIMCLR_DANN_SOURCE_RUN_ROOT / f"fold{fold}" / "resolved_config.yaml"
    if not source_checkpoint.exists():
        raise FileNotFoundError(f"Missing SimCLR+DANN source checkpoint for fold{fold}: {source_checkpoint}")
    if not source_config.exists():
        raise FileNotFoundError(f"Missing SimCLR+DANN source config for fold{fold}: {source_config}")

    run_name = f"{simclr_dann_timestamp}__finetune__benin_to_sa__src-fold{fold}__{SIMCLR_DANN_RUN_TAG}__cfg-{SIMCLR_DANN_CONFIG_NAME}"
    run_root = SCRATCH_ROOT / "runs" / "finetune" / "benin_to_sa" / run_name
    if run_root.exists():
        raise FileExistsError(f"Refusing to reuse an existing SimCLR+DANN run directory: {run_root}")

    launch_cmd = [
        "bash",
        str(FINETUNE_SCRIPT),
        "--source-dataset", "benin",
        "--target-dataset", "sa",
        "--source-checkpoint", str(source_checkpoint),
        "--source-config", str(source_config),
        "--config", str(FINETUNE_BASE_CONFIG),
        "--finetune-preset", SIMCLR_DANN_FINETUNE_PRESET,
        "--clip-unfreeze-last-n-layers", str(SIMCLR_DANN_CLIP_UNFREEZE_LAST_N_LAYERS),
    ]
    if SIMCLR_DANN_FREEZE_BACKBONE:
        launch_cmd.append("--freeze-backbone")
    launch_cmd.extend([
        "--domain-adaptation", "dann",
        "--runtime", SIMCLR_DANN_RUNTIME,
        "--run-name", run_name,
        "--run-root", str(run_root),
    ])
    if SIMCLR_DANN_RUNTIME == "local-venv":
        launch_cmd.extend(["--local-venv", str(SIMCLR_DANN_LOCAL_VENV)])
    for config_set in SIMCLR_DANN_CONFIG_SETS:
        launch_cmd.extend(["--set", config_set])

    print(f"\nLaunching SimCLR+DANN fold{fold}:")
    print(" ".join(shlex.quote(part) for part in launch_cmd))
    completed = subprocess.run(launch_cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"SimCLR+DANN submission failed for fold{fold} with exit code {completed.returncode}")

    run_info = parse_key_value_output(completed.stdout)
    run_info.setdefault("fold", fold)
    run_info.setdefault("run_name", run_name)
    run_info.setdefault("run_root", str(run_root))
    run_info["fold"] = fold
    run_info["source_checkpoint"] = str(source_checkpoint)
    run_info["source_config"] = str(source_config)
    run_info["finetune_preset"] = SIMCLR_DANN_FINETUNE_PRESET
    run_info["domain_adaptation"] = "dann"
    run_info["config_name"] = SIMCLR_DANN_CONFIG_NAME
    run_info["config_sets"] = list(SIMCLR_DANN_CONFIG_SETS)
    run_info["freeze_backbone"] = SIMCLR_DANN_FREEZE_BACKBONE
    run_info["clip_unfreeze_last_n_layers"] = SIMCLR_DANN_CLIP_UNFREEZE_LAST_N_LAYERS
    run_info["runtime"] = SIMCLR_DANN_RUNTIME
    SIMCLR_DANN_RUNS.append(run_info)

print("\nSimCLR+DANN runs")
print(json.dumps(SIMCLR_DANN_RUNS, indent=2))
print("\nIf you restart the kernel, paste these run roots into SIMCLR_DANN_RUN_ROOTS_BY_FOLD in the next path cell.")


## 24. SimCLR+DANN Result Paths

Define the SimCLR+DANN run roots used by the readout and summary cells.


In [ ]:
from pathlib import Path

try:
    SIMCLR_DANN_SOURCE_RUN_ROOT = Path(SIMCLR_SOURCE_RUN_ROOT)
except NameError:
    SIMCLR_DANN_SOURCE_RUN_ROOT = Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/train/benin/20260507_082951__train__benin__all-folds__benin-sa-strong-patientaware-simclr-repro")

# Update these paths after launching fresh SimCLR+DANN finetuning jobs.
SIMCLR_DANN_RUN_ROOTS_BY_FOLD = {
    0: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260508_101625__finetune__benin_to_sa__src-fold0__simclr-dann-target-emphasis-repro__cfg-target_emphasis"),
    1: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260508_101625__finetune__benin_to_sa__src-fold1__simclr-dann-target-emphasis-repro__cfg-target_emphasis"),
    2: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260508_101625__finetune__benin_to_sa__src-fold2__simclr-dann-target-emphasis-repro__cfg-target_emphasis"),
    3: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260508_101625__finetune__benin_to_sa__src-fold3__simclr-dann-target-emphasis-repro__cfg-target_emphasis"),
    4: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260508_101625__finetune__benin_to_sa__src-fold4__simclr-dann-target-emphasis-repro__cfg-target_emphasis"),
}

try:
    SIMCLR_DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD = dict(SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD)
except NameError:
    SIMCLR_DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD = {
        0: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260504__finetune__benin_to_sa__src-fold0__benin-sa-strong-patientaware-simclr-balanceclip2pos4-freezeclip-localvenv-v2"),
        1: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260504__finetune__benin_to_sa__src-fold1__benin-sa-strong-patientaware-simclr-balanceclip2pos4-freezeclip-localvenv-v2"),
        2: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260504__finetune__benin_to_sa__src-fold2__benin-sa-strong-patientaware-simclr-balanceclip2pos4-freezeclip-localvenv-v2"),
        3: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260504__finetune__benin_to_sa__src-fold3__benin-sa-strong-patientaware-simclr-balanceclip2pos4-freezeclip-localvenv-v2"),
        4: Path("/capstor/scratch/cscs/lxflk/ULTR-AI-Vid/runs/finetune/benin_to_sa/20260504__finetune__benin_to_sa__src-fold4__benin-sa-strong-patientaware-simclr-balanceclip2pos4-freezeclip-localvenv-v2"),
    }


## 25. Read SimCLR+DANN Results

Read SimCLR+DANN target and source-retention metrics, using configured SimCLR zero-shot references if needed.


In [ ]:
from pathlib import Path
import csv
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

for required_name in [
    "SIMCLR_DANN_SOURCE_RUN_ROOT",
    "SIMCLR_DANN_RUN_ROOTS_BY_FOLD",
    "SIMCLR_DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD",
]:
    if required_name not in globals():
        raise RuntimeError(f"Run the SimCLR+DANN Result Paths cell before this readout cell. Missing {required_name}.")

SIMCLR_DANN_RESULT_SUBDIR = "dann_full"
SIMCLR_DANN_CONFIG_NAME = "target_emphasis"
SIMCLR_DANN_CONFIG_SETS = [
    'dann_lambda=0.75',
    'dann_domain_warmup_epochs=1',
    'dann_min_best_epoch=2',
    'dann_domain_loss_weight=0.5',
    'dann_source_task_weight=0.5',
    'dann_target_task_weight=2.0',
    'dann_domain_lr=0.00008',
]
SIMCLR_DANN_HISTORICAL_REFERENCE = {
    'zero_shot_auroc_mean': 0.6197,
    'zero_shot_auprc_mean': 0.1129,
    'target_finetuned_auroc_mean': 0.8020,
    'target_finetuned_auprc_mean': 0.3852,
    'source_retention_auroc_mean': 0.7786,
    'source_retention_auprc_mean': 0.6797,
}

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected result file: {path}\n"
            "The corresponding Slurm job is probably still running or failed before evaluation."
        )
    with path.open() as handle:
        return json.load(handle)

def auroc_auprc(payload: dict) -> tuple[float, float]:
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc", metrics.get("auroc")))
    auprc = payload.get("auprc", metrics.get("auprc"))
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find AUROC/AUPRC in payload keys: {sorted(payload.keys())}")
    return float(auroc), float(auprc)

def read_source_model_summary(run_root: Path) -> tuple[float | None, float | None]:
    candidates = [
        run_root / "final_results" / "tb_results" / "tb_test_metrics_summary.csv",
        run_root / "final_results" / "tb_results" / "tb_metrics_summary.csv",
    ]
    summary_csv = next((path for path in candidates if path.exists()), None)
    if summary_csv is None:
        return None, None
    with summary_csv.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 1:
        raise ValueError(f"Expected one source summary row in {summary_csv}, found {len(rows)}")
    row = rows[0]
    auroc = row.get("auc_mean", row.get("auroc_mean"))
    auprc = row.get("auprc_mean")
    if auroc is None or auprc is None:
        raise ValueError(f"Could not find source AUROC/AUPRC columns in {summary_csv}: {sorted(row)}")
    return float(auroc), float(auprc)

def mean_std(rows: list[dict], key: str) -> tuple[float, float]:
    values = [float(row[key]) for row in rows]
    return sum(values) / len(values), stats.stdev(values) if len(values) > 1 else 0.0

def fmt_pair(auroc, auprc):
    if auroc is None or auprc is None:
        return "n/a"
    return f"{auroc:.4f}/{auprc:.4f}"

rows = []
for fold, run_root in sorted(SIMCLR_DANN_RUN_ROOTS_BY_FOLD.items()):
    target_payload = load_json(run_root / "results" / SIMCLR_DANN_RESULT_SUBDIR / "results.json")
    retention_payload = load_json(run_root / "results" / "source_test_evaluation" / "source_test_results_finetuned.json")

    zero_shot_path = run_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
    zero_reference_used = False
    if not zero_shot_path.exists():
        reference_root = SIMCLR_DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD.get(fold)
        if reference_root is None:
            raise FileNotFoundError(f"Missing zero-shot result and no reference root configured for fold{fold}: {zero_shot_path}")
        zero_shot_path = reference_root / "results" / "source_zero_shot" / "source_zero_shot_results.json"
        zero_reference_used = True
    zero_payload = load_json(zero_shot_path)

    zero_auroc, zero_auprc = auroc_auprc(zero_payload)
    target_auroc, target_auprc = auroc_auprc(target_payload)
    retention_auroc, retention_auprc = auroc_auprc(retention_payload)

    rows.append({
        "fold": fold,
        "run_root": str(run_root),
        "result_subdir": SIMCLR_DANN_RESULT_SUBDIR,
        "zero_shot_reference_used": zero_reference_used,
        "zero_shot_auroc": zero_auroc,
        "zero_shot_auprc": zero_auprc,
        "target_finetuned_auroc": target_auroc,
        "target_finetuned_auprc": target_auprc,
        "source_retention_auroc": retention_auroc,
        "source_retention_auprc": retention_auprc,
    })

summary = {}
for key in [
    "zero_shot_auroc",
    "zero_shot_auprc",
    "target_finetuned_auroc",
    "target_finetuned_auprc",
    "source_retention_auroc",
    "source_retention_auprc",
]:
    mean_value, std_value = mean_std(rows, key)
    summary[f"{key}_mean"] = mean_value
    summary[f"{key}_std"] = std_value

source_auroc, source_auprc = read_source_model_summary(SIMCLR_DANN_SOURCE_RUN_ROOT)
summary.update({
    "source_model_auroc": source_auroc,
    "source_model_auprc": source_auprc,
    "domain_adaptation": "simclr+dann",
    "config_name": SIMCLR_DANN_CONFIG_NAME,
    "config_sets": "; ".join(SIMCLR_DANN_CONFIG_SETS),
    "finetune_preset": "balance_clip2_pos4_lowlr",
    "freeze_backbone": True,
    "clip_unfreeze_last_n_layers": 2,
    "result_subdir": SIMCLR_DANN_RESULT_SUBDIR,
    "n_folds": len(rows),
    "zero_shot_reference_folds": sum(1 for row in rows if row["zero_shot_reference_used"]),
})

for key, expected in SIMCLR_DANN_HISTORICAL_REFERENCE.items():
    summary[f"{key}_historical"] = expected
    summary[f"{key}_delta"] = summary[key] - expected

SIMCLR_DANN_TRANSFER_ROWS = rows
SIMCLR_DANN_TRANSFER_SUMMARY = summary

if pd is not None:
    display(pd.DataFrame(rows))
    display(pd.DataFrame([summary]))
else:
    print(json.dumps(rows, indent=2))
    print(json.dumps(summary, indent=2))

print(
    "SimCLR+DANN summary: "
    f"source {fmt_pair(summary['source_model_auroc'], summary['source_model_auprc'])}; "
    f"zero-shot SA {summary['zero_shot_auroc_mean']:.4f}/{summary['zero_shot_auprc_mean']:.4f}; "
    f"DANN-finetuned SA {summary['target_finetuned_auroc_mean']:.4f}/{summary['target_finetuned_auprc_mean']:.4f}; "
    f"source retention {summary['source_retention_auroc_mean']:.4f}/{summary['source_retention_auprc_mean']:.4f}"
)


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultrai.analysis.domain_shift.metric_curve_plots import plot_run_roots_roc_pr

simclr_dann_curve_summary = plot_run_roots_roc_pr(
    SIMCLR_DANN_RUN_ROOTS_BY_FOLD,
    'results',
    SIMCLR_DANN_RESULT_SUBDIR,
    f"{SIMCLR_DANN_RESULT_SUBDIR}_patient_predictions.csv",
    evaluation_label='SimCLR + DANN',
    title='SimCLR + DANN Test Set Performance',
    subtitle="Mean ROC and Precision-Recall Curves Across Five Source Folds",
    output_filename='simclr_dann_sa_target_roc_pr_mean_ci.svg',
    color='#2ca02c',
)

print(f"Saved SimCLR + DANN AUROC/AUPRC graph to {simclr_dann_curve_summary['output_svg']}")
print(
    'SimCLR + DANN' + " curves: "
    f"AUROC {simclr_dann_curve_summary['auroc']['mean']:.4f} "
    f"(95% CI {simclr_dann_curve_summary['auroc']['ci_lower']:.4f}-"
    f"{simclr_dann_curve_summary['auroc']['ci_upper']:.4f}); "
    f"AUPRC {simclr_dann_curve_summary['auprc']['mean']:.4f} "
    f"(95% CI {simclr_dann_curve_summary['auprc']['ci_lower']:.4f}-"
    f"{simclr_dann_curve_summary['auprc']['ci_upper']:.4f})"
)


## 26. Summary Table

Build a compact AUROC/AUPRC table from the available run paths and readout variables; missing results are shown as `n/a`.


In [ ]:
from pathlib import Path
import csv
import json
import statistics as stats

try:
    pd
except NameError:
    try:
        import pandas as pd
    except ImportError:
        pd = None

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        display = print

EXPECTED_FOLDS = tuple(range(5))

def maybe_float(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def as_path(value):
    if value is None:
        return None
    path = Path(value)
    if "PASTE" in str(path):
        return None
    return path

def get_path_var(name: str):
    return as_path(globals().get(name))

def get_roots_var(name: str):
    raw = globals().get(name)
    if not isinstance(raw, dict):
        return {}
    roots = {}
    for fold, value in raw.items():
        try:
            fold_id = int(fold)
        except (TypeError, ValueError):
            continue
        path = as_path(value)
        if path is not None:
            roots[fold_id] = path
    return roots

def load_json_if_exists(path: Path | None):
    if path is None or not path.exists():
        return None
    try:
        with path.open() as handle:
            return json.load(handle)
    except Exception:
        return None

def auroc_auprc(payload: dict | None):
    if not payload:
        return None, None
    metrics = payload.get("test_metrics", {})
    auroc = payload.get("auroc", metrics.get("auc", metrics.get("auroc")))
    auprc = payload.get("auprc", metrics.get("auprc"))
    return maybe_float(auroc), maybe_float(auprc)

def mean_pair(pairs):
    if not pairs:
        return None, None
    if any(auroc is None or auprc is None for auroc, auprc in pairs):
        return None, None
    return (
        sum(auroc for auroc, _ in pairs) / len(pairs),
        sum(auprc for _, auprc in pairs) / len(pairs),
    )

def read_source_pair_from_summary(run_root: Path | None):
    if run_root is None:
        return None, None
    candidates = [
        run_root / "final_results" / "tb_results" / "tb_test_metrics_summary.csv",
        run_root / "final_results" / "tb_results" / "tb_metrics_summary.csv",
    ]
    summary_csv = next((path for path in candidates if path.exists()), None)
    if summary_csv is None:
        return None, None
    try:
        with summary_csv.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
    except Exception:
        return None, None
    if not rows:
        return None, None
    row = rows[0]
    return maybe_float(row.get("auc_mean", row.get("auroc_mean"))), maybe_float(row.get("auprc_mean"))

def source_pair_from_result(result_name: str, root_var_name: str, fallback_summary: dict | None = None):
    result = globals().get(result_name)
    if isinstance(result, dict):
        auroc = maybe_float(result.get("mean_auroc", result.get("source_model_auroc")))
        auprc = maybe_float(result.get("mean_auprc", result.get("source_model_auprc")))
        if auroc is not None and auprc is not None:
            return auroc, auprc
    if isinstance(fallback_summary, dict):
        auroc = maybe_float(fallback_summary.get("source_model_auroc"))
        auprc = maybe_float(fallback_summary.get("source_model_auprc"))
        if auroc is not None and auprc is not None:
            return auroc, auprc
    return read_source_pair_from_summary(get_path_var(root_var_name))

def pair_from_summary(summary: dict | None, auroc_key: str, auprc_key: str):
    if not isinstance(summary, dict):
        return None, None
    return maybe_float(summary.get(auroc_key)), maybe_float(summary.get(auprc_key))

def summary_from_rows(rows):
    if not isinstance(rows, list) or len(rows) != len(EXPECTED_FOLDS):
        return {}
    summary = {}
    for key in [
        "zero_shot_auroc", "zero_shot_auprc",
        "target_finetuned_auroc", "target_finetuned_auprc",
        "source_retention_auroc", "source_retention_auprc",
    ]:
        values = [maybe_float(row.get(key)) for row in rows if isinstance(row, dict)]
        summary[f"{key}_mean"] = sum(values) / len(values) if len(values) == len(EXPECTED_FOLDS) and all(value is not None for value in values) else None
    return summary

def metric_pair_from_roots(roots_by_fold: dict[int, Path], relative_parts: tuple[str, ...], fallback_roots_by_fold: dict[int, Path] | None = None):
    if set(roots_by_fold) != set(EXPECTED_FOLDS):
        return None, None
    pairs = []
    for fold in EXPECTED_FOLDS:
        path = roots_by_fold[fold].joinpath(*relative_parts)
        if not path.exists() and fallback_roots_by_fold:
            fallback_root = fallback_roots_by_fold.get(fold)
            path = fallback_root.joinpath(*relative_parts) if fallback_root is not None else None
        pairs.append(auroc_auprc(load_json_if_exists(path)))
    return mean_pair(pairs)

def transfer_summary_from_roots(roots_by_fold: dict[int, Path], result_subdir: str, zero_reference_roots_by_fold: dict[int, Path] | None = None):
    zero = metric_pair_from_roots(
        roots_by_fold,
        ("results", "source_zero_shot", "source_zero_shot_results.json"),
        zero_reference_roots_by_fold,
    )
    target = metric_pair_from_roots(
        roots_by_fold,
        ("results", result_subdir, "results.json"),
    )
    retention = metric_pair_from_roots(
        roots_by_fold,
        ("results", "source_test_evaluation", "source_test_results_finetuned.json"),
    )
    return {
        "zero_shot_auroc_mean": zero[0],
        "zero_shot_auprc_mean": zero[1],
        "target_finetuned_auroc_mean": target[0],
        "target_finetuned_auprc_mean": target[1],
        "source_retention_auroc_mean": retention[0],
        "source_retention_auprc_mean": retention[1],
    }

def get_transfer_summary(summary_name: str, rows_name: str, roots_name: str, result_subdir: str, zero_reference_roots_name: str | None = None):
    summary = globals().get(summary_name)
    if isinstance(summary, dict):
        return summary
    rows_summary = summary_from_rows(globals().get(rows_name))
    if rows_summary:
        return rows_summary
    roots = get_roots_var(roots_name)
    zero_reference_roots = get_roots_var(zero_reference_roots_name) if zero_reference_roots_name else None
    if not roots:
        return {}
    return transfer_summary_from_roots(roots, result_subdir, zero_reference_roots)

def fmt_pair(auroc, auprc):
    if auroc is None or auprc is None:
        return "n/a"
    return f"{auroc:.4f} / {auprc:.4f}"

def add_row(rows, method, source_pair=(None, None), zero_pair=(None, None), finetuned_pair=(None, None), retention_pair=(None, None)):
    rows.append({
        "Method": method,
        "Source": fmt_pair(*source_pair),
        "Zero-Shot SA": fmt_pair(*zero_pair),
        "Finetuned SA": fmt_pair(*finetuned_pair),
        "Source Retention": fmt_pair(*retention_pair),
        "Source AUROC": source_pair[0],
        "Source AUPRC": source_pair[1],
        "Zero-Shot SA AUROC": zero_pair[0],
        "Zero-Shot SA AUPRC": zero_pair[1],
        "Finetuned SA AUROC": finetuned_pair[0],
        "Finetuned SA AUPRC": finetuned_pair[1],
        "Source Retention AUROC": retention_pair[0],
        "Source Retention AUPRC": retention_pair[1],
    })

baseline_source = source_pair_from_result("source_result", "SOURCE_BASELINE_RUN_ROOT")
normal_summary = get_transfer_summary(
    "NORMAL_FINETUNE_SUMMARY",
    "normal_ft_rows",
    "NORMAL_FINETUNE_RUN_ROOTS_BY_FOLD",
    "finetune_full",
)
normal_zero = pair_from_summary(normal_summary, "zero_shot_auroc_mean", "zero_shot_auprc_mean")
normal_finetuned = pair_from_summary(normal_summary, "target_finetuned_auroc_mean", "target_finetuned_auprc_mean")
normal_retention = pair_from_summary(normal_summary, "source_retention_auroc_mean", "source_retention_auprc_mean")

simclr_summary = get_transfer_summary(
    "SIMCLR_TRANSFER_SUMMARY",
    "SIMCLR_TRANSFER_ROWS",
    "SIMCLR_FINETUNE_RUN_ROOTS_BY_FOLD",
    "finetune_full",
)
simclr_source = source_pair_from_result("simclr_source_result", "SIMCLR_SOURCE_RUN_ROOT", simclr_summary)
simclr_zero = pair_from_summary(simclr_summary, "zero_shot_auroc_mean", "zero_shot_auprc_mean")
simclr_finetuned = pair_from_summary(simclr_summary, "target_finetuned_auroc_mean", "target_finetuned_auprc_mean")
simclr_retention = pair_from_summary(simclr_summary, "source_retention_auroc_mean", "source_retention_auprc_mean")

summary_rows = []
add_row(
    summary_rows,
    "Benin baseline + plain finetuning",
    baseline_source,
    normal_zero,
    normal_finetuned,
    normal_retention,
)
add_row(
    summary_rows,
    "SimCLR backbone + finetuning",
    simclr_source,
    simclr_zero,
    simclr_finetuned,
    simclr_retention,
)

for method, upper, source_root_var, roots_var, zero_ref_var, result_subdir in [
    ("SimCLR backbone + DANN", "SIMCLR_DANN", "SIMCLR_DANN_SOURCE_RUN_ROOT", "SIMCLR_DANN_RUN_ROOTS_BY_FOLD", "SIMCLR_DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD", "dann_full"),
    ("DANN", "DANN", "DANN_SOURCE_RUN_ROOT", "DANN_RUN_ROOTS_BY_FOLD", "DANN_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD", "dann_full"),
    ("FixMatch", "FIXMATCH", "FIXMATCH_SOURCE_RUN_ROOT", "FIXMATCH_RUN_ROOTS_BY_FOLD", "FIXMATCH_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD", "fixmatch_full"),
    ("EWC", "EWC", "EWC_SOURCE_RUN_ROOT", "EWC_RUN_ROOTS_BY_FOLD", "EWC_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD", "ewc_full"),
    ("LwF", "LWF", "LWF_SOURCE_RUN_ROOT", "LWF_RUN_ROOTS_BY_FOLD", "LWF_ZERO_SHOT_REFERENCE_RUN_ROOTS_BY_FOLD", "lwf_full"),
]:
    method_summary = get_transfer_summary(
        f"{upper}_TRANSFER_SUMMARY",
        f"{upper}_TRANSFER_ROWS",
        roots_var,
        result_subdir,
        zero_ref_var,
    )
    add_row(
        summary_rows,
        method,
        source_pair_from_result("", source_root_var, method_summary),
        pair_from_summary(method_summary, "zero_shot_auroc_mean", "zero_shot_auprc_mean"),
        pair_from_summary(method_summary, "target_finetuned_auroc_mean", "target_finetuned_auprc_mean"),
        pair_from_summary(method_summary, "source_retention_auroc_mean", "source_retention_auprc_mean"),
    )

RESULTS_SUMMARY_ROWS = summary_rows
compact_columns = ["Method", "Source", "Zero-Shot SA", "Finetuned SA", "Source Retention"]
if pd is not None:
    RESULTS_SUMMARY_TABLE = pd.DataFrame(summary_rows)
    display(RESULTS_SUMMARY_TABLE[compact_columns])
else:
    RESULTS_SUMMARY_TABLE = summary_rows
    print(json.dumps([{key: row[key] for key in compact_columns} for row in summary_rows], indent=2))
